# 01 · Datos y Feature Store — Adult Income

**Business case:** banca/marketing — identificar clientes potenciales con probabilidad de superar los USD 50K de ingreso anual, para priorizar la oferta de un producto crediticio premium.

**Objetivo:** cargar Adult Income (OpenML), guardar una tabla Delta raw y publicar seis variables en un Feature Store gobernado por Unity Catalog, luego evolucionar el esquema agregando una 7ª feature derivada.

> Adjunta este notebook a compute Serverless en Databricks Free Edition. A diferencia de Breast Cancer, este dataset SÍ requiere descarga externa desde OpenML (ya validado que el clúster tiene salida a internet).

Todos los nombres editables están en la primera celda de configuración.


## 1. Configuración

In [0]:
%pip install databricks-feature-engineering --quiet

dbutils.library.restartPython()


In [0]:
from sklearn.datasets import fetch_openml

# --- Dataset Adult Income (OpenML) ---
adult = fetch_openml(name="adult", version=2, as_frame=True)
df_adult = adult.frame.copy()

# --- Catálogo y esquema ---
spark.sql("CREATE SCHEMA IF NOT EXISTS mlops_income_course")

current_catalog = spark.catalog.currentCatalog()
SCHEMA = "mlops_income_course"

income_raw = f"{current_catalog}.{SCHEMA}.income_raw"
income_features = f"{current_catalog}.{SCHEMA}.income_features"

print(f"Catálogo: {current_catalog}")
print(f"Esquema:  {SCHEMA}")
print(f"Tabla raw:      {income_raw}")
print(f"Feature Table:  {income_features}")
print(f"\nDataset cargado: {df_adult.shape[0]} filas, {df_adult.shape[1]} columnas")
df_adult.head(10)


## 2. Datos mínimos

Adult Income solo tiene 6 columnas numéricas nativas (el resto son categóricas y no se usan, para mantener la misma simplicidad que el ejemplo de referencia). El target real se llama `class` (valores `>50K` / `<=50K`); lo convertimos a `target` binario. `client_id` será la clave de la entidad.


In [0]:
from sklearn.datasets import fetch_openml

# --- Cargar dataset como DataFrame de pandas ---
adult = fetch_openml(name="adult", version=2, as_frame=True)
df_pandas = adult.frame[[
    "age", "education-num", "hours-per-week",
    "capital-gain", "capital-loss", "fnlwgt", "class",
]].copy()

# --- Renombrar columnas con guion a guion bajo ---
rename_map = {
    "education-num": "education_num",
    "hours-per-week": "hours_per_week",
    "capital-gain": "capital_gain",
    "capital-loss": "capital_loss",
}
df_pandas = df_pandas.rename(columns=rename_map)

# --- Construir target binario a partir de class (1 = >50K, 0 = <=50K) ---
df_pandas["target"] = (
    df_pandas["class"].astype(str).str.strip().eq(">50K").astype(int)
)
df_pandas = df_pandas.drop(columns=["class"])

# --- Agregar client_id consecutivo desde 1 ---
df_pandas.insert(0, "client_id", range(1, len(df_pandas) + 1))

# --- Convertir a Spark DataFrame ---
df_income = spark.createDataFrame(df_pandas)

df_income.show(10, truncate=False)
print(f"Total de filas: {df_income.count()}")
print(f"Distribución de target:")
df_income.groupBy("target").count().show()


## 3. Capa raw en Delta

La tabla raw conserva features, identificador y etiqueta. Sobrescribirla permite repetir la práctica sin crear duplicados.


In [0]:
df_income.write \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(income_raw)

print(f"Tabla guardada: {income_raw}")

spark.read.table(income_raw).show(10, truncate=False)


## 4. Feature Table

La Feature Table contiene la entidad y sus variables reutilizables, pero **no la etiqueta**. La clave primaria (`client_id`) permite relacionarla después con el DataFrame de labels.

La celda es idempotente: crea la tabla la primera vez y la actualiza en ejecuciones posteriores.


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# --- Seleccionar client_id y las seis features desde la tabla raw ---
feature_cols = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss", "fnlwgt"]
features_df = spark.read.table(income_raw).select("client_id", *feature_cols)

# --- Crear o actualizar la Feature Table ---
try:
    fe.write_table(
        name=income_features,
        df=features_df,
        mode="merge",
    )
    print(f"✅ Feature Table actualizada por merge: {income_features}")
except Exception:
    fe.create_table(
        name=income_features,
        primary_keys=["client_id"],
        df=features_df,
        schema=features_df.schema,
        description=(
            "Feature Table del dataset Adult Income: seis variables numéricas por cliente "
            "(edad, años de educación, horas semanales, ganancia y pérdida de capital, peso censal) "
            "para scoring de producto crediticio premium."
        ),
    )
    print(f"✅ Feature Table creada: {income_features}")


### Leer la Feature Table

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

fe.read_table(name=income_features).show(10, truncate=False)


### Calcular una feature derivada

`capital_net_ratio` = ganancia de capital neta relativa a horas trabajadas: (capital_gain − capital_loss) / (hours_per_week + 1). El +1 evita división entre cero.


In [0]:
from pyspark.sql.functions import col

# --- Calcular feature derivada: (capital_gain - capital_loss) / (hours_per_week + 1) ---
df_ratio = df_income.select(
    col("client_id"),
    (
        (col("capital_gain") - col("capital_loss")) / (col("hours_per_week") + 1)
    ).alias("capital_net_ratio"),
)

df_ratio.show(10, truncate=False)


### Evolucionar el Feature Store

Agregamos `capital_net_ratio` a la Feature Table existente mediante merge, sin recrearla.


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.functions import col

fe = FeatureEngineeringClient()

feature_cols = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss", "fnlwgt"]

features_df = (
    spark.read.table(income_raw)
    .select("client_id", *feature_cols)
    .join(df_ratio, on="client_id", how="inner")  # trae capital_net_ratio
)

fe.write_table(
    name=income_features,
    df=features_df,
    mode="merge",
)

print(f"✅ Feature actualizada con capital_net_ratio: {income_features}")


### Verificar la nueva feature

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

fe.read_table(name=income_features) \
    .select("client_id", "capital_gain", "capital_loss", "hours_per_week", "capital_net_ratio") \
    .show(10, truncate=False)


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {income_features}"))


## Cierre

```text
OpenML (Adult Income) → Spark DataFrame → Delta raw (income_raw)
                                        └→ Feature Table (income_features, client_id como clave)
                                           └→ evolución: + capital_net_ratio
```

**Comprueba:** en Catalog Explorer deben aparecer `income_raw` e `income_features`. Continúa con el notebook 02.
